# VAE : d'un espace latent troue a un espace latent echantillonnable

Le notebook `03_autoencoder` se terminait sur un aveu. Pour generer des images, il fallait
**ajuster apres coup** une gaussienne sur les codes observes (`sample_gaussian_latent`), parce
que rien dans l'entrainement d'un AutoEncoder ne dit ou vivent les codes. Une bonne part des
tirages tombait donc hors de la zone ou le decodeur avait ete entraine, et les images
generees s'en ressentaient.

L'AutoEncodeur Variationnel s'attaque exactement a ce probleme. Son encodeur ne rend plus un
point mais une **distribution** `q(z|x) = N(mu(x), diag(exp(logvar(x))))`, et sa perte ajoute
un terme **KL** qui pousse cette distribution vers un prior connu d'avance, `N(0, I)`.

Consequence: on n'ajuste plus rien. On echantillonne `N(0, I)` directement.

Ce notebook reprend les memes experiences que `03_autoencoder` (dimension latente, activation
de sortie x perte, epochs, profondeur, formes) et y ajoute ce qui n'a de sens que pour un VAE:
la preuve que le KL a fonctionne, le balayage de `beta`, et le diagnostic de collapse.

In [ ]:
from src.dataset import load_mnist_dataset, load_shapes_npz
from src.autoencoder import AutoEncoder
from src.vae import VariationalAutoEncoder
from src.helper import extract_full_dataset, get_device
from src.metrics import compression_report, Latent
from src.viz import (
    finish_figure,
    format_fixed_note,
    show_original_vs_reconstruction_grid,
    show_image_grid,
    show_labeled_image_rows,
    plot_latent_scatter,
    print_compression_report,
    sample_gaussian_latent,
    generate_from_latent_using_gaussian,
    interpolate_latent,
    subsample_dataset,
)

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)

print("device:", get_device())

EPOCHS = 40
EPOCHS_SWEEP = 40
BATCH_SIZE = 128

In [ ]:
def describe_vae(model, n_train=None, omit=(), extra=None):
    """Note du bas des figures, lue sur le modele entraine.

    describe_autoencoder ne marche PAS ici: l'encodeur d'un VAE n'est pas un Sequential
    plat, `model.encoder.net` n'en est que le tronc. Il rapporterait couches=784-107-14,
    en perdant la dimension latente, et leverait une IndexError a la profondeur 1.
    """
    trunk_linears = [layer for layer in model.encoder.net if isinstance(layer, nn.Linear)]
    if trunk_linears:
        sizes = [trunk_linears[0].in_features] + [layer.out_features for layer in trunk_linears]
    else:
        sizes = [model.encoder.mu_head.in_features]   # profondeur 1: pas de tronc
    sizes.append(model.encoder.mu_head.out_features)  # la tete latente ferme la liste
    # Pas besoin du garde-fou "last_linear" de l'AE: le VAE n'expose pas de latent_activation,
    # donc la premiere non-Linear du tronc EST l'activation cachee.
    activation = next((type(layer).__name__ for layer in model.encoder.net
                       if not isinstance(layer, nn.Linear)), "aucune")
    fields = {
        "couches": "-".join(str(size) for size in sizes) + " (2 tetes)",
        "act": activation,
        "out_act": model.output_activation.__name__ if model.output_activation else "aucune",
        # (sum) et beta sont affiches: la reduction est le piege central de ce notebook
        "loss": f"{model.fonction_loss.__name__}(sum)+{model.beta}*KL",
        "epochs": str(len(model.loss_history)),
    }
    if n_train is not None:
        fields["n_train"] = f"{n_train:,}"
    return format_fixed_note(fields, omit, extra)


def run_vae_experiment(
        X_train, X_eval, input_dim, latent_dim, activation_function, epochs,
        loss_function=nn.BCELoss, beta=1.0, output_activation_function=nn.Sigmoid,
        encoder_layer_num=3, decoder_layer_num=3, learning_rate=1e-3, seed=None,
        ):
    """Pendant VAE de run_autoencoder_hyperparam_experiment. Fonction distincte car la
    signature differe: beta en plus, latent_activation en moins (incoherent pour un VAE)."""
    if seed is not None:
        torch.manual_seed(seed)
    model = VariationalAutoEncoder(
        input_dim=input_dim, output_dim=input_dim, latent_dim=latent_dim,
        encoder_layer_num=encoder_layer_num, decoder_layer_num=decoder_layer_num,
        encoder_activation_function=activation_function, loss_function=loss_function,
        beta=beta, output_activation_function=output_activation_function,
    )
    model.fit(X_train, epochs=epochs, batch_size=BATCH_SIZE, learning_rate=learning_rate)
    latent = model.encode(X_eval)                 # mu, deterministe
    reconstruction = model.decode(latent)
    report = compression_report(model.get_codebook(), latent, X_eval, reconstruction)
    return {"model": model, "latent": latent, "reconstruction": reconstruction, "report": report}


def posterior_stats(model, X_eval, seuil_actif=0.01):
    """Statistiques du posterior q(z|x): KL par dimension et dimensions actives.

    Une dimension dont le KL est ~0 est *collapsee*: l'encodeur y rend exactement le prior,
    elle ne transporte aucune information et le decodeur l'ignore.
    """
    mu, logvar = model.encode_distribution(X_eval)
    kl_per_dim = model.kl_divergence_per_dim(
        torch.from_numpy(mu), torch.from_numpy(logvar)
    ).numpy()
    return {
        "mu": mu,
        "sigma": np.exp(0.5 * logvar),
        "kl_per_dim": kl_per_dim,
        "kl_total": float(kl_per_dim.sum()),
        "dims_actives": int((kl_per_dim > seuil_actif).sum()),
    }


def prior_gap(codes):
    """Distance du nuage de codes au prior N(0, I): a quel point le KL a-t-il travaille.

    ||E[mu]|| mesure le decentrage, ||Cov(mu) - I||_F la deformation. Les deux tendent vers 0
    si le latent suit vraiment le prior.
    """
    centre = float(np.linalg.norm(codes.mean(axis=0)))
    ecart_cov = float(np.linalg.norm(np.cov(codes, rowvar=False) - np.eye(codes.shape[1])))
    return centre, ecart_cov

## Partie A - MNIST DIGITS

In [ ]:
mnist_train_images, mnist_train_labels = extract_full_dataset(load_mnist_dataset(train=True, shuffle=False))
mnist_eval_images, mnist_eval_labels = extract_full_dataset(load_mnist_dataset(train=False, shuffle=False))

MNIST_SHAPE = (1, 28, 28)
X_mnist_train, y_mnist_train = subsample_dataset(
    mnist_train_images.reshape(len(mnist_train_images), -1).numpy(), mnist_train_labels.numpy(), 15000
    )
X_mnist_eval, y_mnist_eval = subsample_dataset(
    mnist_eval_images.reshape(len(mnist_eval_images), -1).numpy(), mnist_eval_labels.numpy(), 3000
    )
X_tr, y_tr = subsample_dataset(X_mnist_train, y_mnist_train, 6000, seed=1)
print("train:", X_mnist_train.shape, "| eval:", X_mnist_eval.shape, "| sweeps:", X_tr.shape)

### Train - le VAE et son jumeau AutoEncoder

On entraine deux modeles **strictement comparables**: meme architecture (3+3 couches, ReLU,
latent 2), meme budget, meme graine. Seule difference: le VAE encode une distribution et paie
un KL.

Le VAE prend `BCELoss` par defaut, et ce n'est pas un detail: pour des pixels dans [0,1]
derriere une sortie Sigmoid, la BCE **est** la log-vraisemblance de Bernoulli, donc le terme
de reconstruction naturel de la borne variationnelle (ELBO). L'AE de reference garde `MSELoss`
comme dans `03`; c'est justement un des axes qu'on rebalayera plus bas.

In [ ]:
GRAINE = 0

vae_run = run_vae_experiment(X_tr, X_mnist_eval, 784, 2, nn.ReLU, EPOCHS,
                             loss_function=nn.BCELoss, beta=1.0, seed=GRAINE)
vae_model = vae_run["model"]

# Le jumeau AE: meme architecture, meme graine, meme budget.
torch.manual_seed(GRAINE)
ae_model = AutoEncoder(input_dim=784, output_dim=784, latent_dim=2,
                       encoder_layer_num=3, decoder_layer_num=3,
                       encoder_activation_function=nn.ReLU, loss_function=nn.MSELoss)
ae_model.fit(X_tr, epochs=EPOCHS, batch_size=BATCH_SIZE)
ae_latent = ae_model.encode(X_mnist_eval)
ae_reconstruction = ae_model.decode(ae_latent)

VAE_CONFIG = describe_vae(vae_model, n_train=len(X_tr))
print(VAE_CONFIG)

# La perte du VAE se lit en deux morceaux: reconstruction et KL n'ont pas la meme unite,
# et c'est leur EQUILIBRE qui compte, pas leur somme.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(vae_model.recon_history, marker=".", label="reconstruction (BCE somme)")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE par image"); axes[0].legend()
axes[0].set_title("Terme de reconstruction")
axes[1].plot(vae_model.kl_history, marker=".", color="tab:orange", label="KL(q(z|x) || N(0,I))")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("KL (nats)"); axes[1].legend()
axes[1].set_title("Terme KL")
finish_figure(fig, "VAE: les deux termes de l'ELBO",
              describe_vae(vae_model, n_train=len(X_tr), omit=("epochs",)))

### Compression et decompression

Le protocole est identique a celui de l'AutoEncoder: l'emetteur envoie `mu`, soit
`latent_dim x 4` octets par image, et `get_codebook()` renvoie les poids du **decodeur**. La
tete `logvar` reste chez l'emetteur: le recepteur n'en a pas besoin. Les deux modeles sont
donc compares a **debit strictement egal**.

Une mise en garde a lire avant le tableau: a dimension latente egale, la MSE du VAE sera
**moins bonne** que celle de l'AE. C'est attendu, la capacite paie le KL. Mais
`compression_report` ne mesure que la compression: **ce que le VAE achete - un espace
echantillonnable - lui est totalement invisible**. Le tableau seul dit "le VAE est moins bon";
la figure de generation, deux cellules plus bas, dit exactement l'inverse.

In [ ]:
vae_latent, vae_reconstruction = vae_run["latent"], vae_run["reconstruction"]

print("=== AutoEncoder (MSE) ===")
ae_report = compression_report(ae_model.get_codebook(), ae_latent, X_mnist_eval, ae_reconstruction)
print_compression_report(ae_report)
print("\n=== VAE (BCE + KL) ===")
print_compression_report(vae_run["report"])

print(f"\nMessage par image: {vae_latent.n_bytes / len(X_mnist_eval):.0f} octets dans les deux cas "
      f"({vae_latent.array.shape[1]} float32) -> debit identique, comparaison honnete.")

show_labeled_image_rows(
    [X_mnist_eval, ae_reconstruction, vae_reconstruction],
    MNIST_SHAPE,
    ["original",
     f"AE\n{ae_report['reconstruction_mse']:.4f}",
     f"VAE\n{vae_run['report']['reconstruction_mse']:.4f}"],
    n=8, title="MNIST: decompression AE vs VAE (etiquette = MSE d'evaluation)",
    config=describe_vae(vae_model, n_train=len(X_tr), omit=("loss",)),
)

### Preuve 1/2 : la geometrie de l'espace latent

Si le KL a fait son travail, le nuage des `mu` doit ressembler a `N(0, I)`: centre a l'origine
et de covariance identite. Deux nombres le mesurent:

- `||E[mu]||` : le decentrage. Zero si le nuage est centre.
- `||Cov(mu) - I||_F` : la deformation. Zero si la covariance vaut l'identite.

Les cercles a 1 et 2 ecarts-types du prior sont traces par-dessus les deux nuages. Sur l'AE ils
n'ont aucun rapport avec les donnees - c'est precisement pourquoi il fallait ajuster une
gaussienne. Sur le VAE ils doivent tomber juste.

In [ ]:
vae_stats = posterior_stats(vae_model, X_mnist_eval)
ae_centre, ae_gap = prior_gap(ae_latent.array)
vae_centre, vae_gap = prior_gap(vae_stats["mu"])

print(f"{'':>12} | {'||E[mu]||':>10} | {'||Cov-I||_F':>12}")
print(f"{'AutoEncoder':>12} | {ae_centre:>10.3f} | {ae_gap:>12.3f}")
print(f"{'VAE':>12} | {vae_centre:>10.3f} | {vae_gap:>12.3f}")
print(f"\nKL total du VAE: {vae_stats['kl_total']:.3f} nats "
      f"| par dimension: {np.round(vae_stats['kl_per_dim'], 3)}")
print(f"sigma moyen par dimension: {np.round(vae_stats['sigma'].mean(axis=0), 3)} "
      f"(le prior impose 1.0; plus petit = la dimension transporte de l'information)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
for ax, (nom, codes, ecart) in zip(axes, [("AutoEncoder", ae_latent.array, ae_gap),
                                          ("VAE", vae_stats["mu"], vae_gap)]):
    scatter = ax.scatter(codes[:, 0], codes[:, 1], c=y_mnist_eval, cmap="tab10", s=4, alpha=0.6)
    for rayon in (1, 2):
        ax.add_patch(plt.Circle((0, 0), rayon, fill=False, color="black",
                                linestyle="--", linewidth=1.2, zorder=5))
    ax.set_title(f"{nom} (||Cov-I||={ecart:.2f})", fontsize=10)
    ax.set_xlabel("z1"); ax.set_ylabel("z2")
    ax.set_aspect("equal", adjustable="datalim")
fig.colorbar(scatter, ax=list(axes), label="chiffre")
finish_figure(fig, "Espace latent vs prior N(0,I) (cercles = 1 et 2 ecarts-types du prior)",
              describe_vae(vae_model, n_train=len(X_tr), omit=("loss",)), layout=False)

### Preuve 2/2 : generer sans rien ajuster

C'est la figure centrale du notebook. Deux modeles, deux facons d'echantillonner:

- **gaussienne ajustee** (`sample_gaussian_latent`): on regarde les codes, on estime moyenne et
  covariance, on tire. C'est le pis-aller du notebook `03`.
- **prior N(0,I)** (`sample_prior`): on tire sans jamais regarder les donnees.

Pour l'AutoEncoder, la ligne "prior" doit etre du bruit: rien n'a jamais force ses codes a
vivre autour de l'origine avec une variance unite. Pour le VAE, les deux lignes doivent se
ressembler - et c'est tout l'argument: **le VAE rend l'ajustement inutile, parce qu'il connait
deja la loi de son latent**.

In [ ]:
N_GEN = 8
lignes, etiquettes = [], []
for nom, modele, latent in [("AE", ae_model, ae_latent), ("VAE", vae_model, vae_latent)]:
    lignes.append(generate_from_latent_using_gaussian(modele, latent, N_GEN, seed=1))
    etiquettes.append(f"{nom}\ngaussienne ajustee")
    if nom == "VAE":
        codes_prior = modele.sample_prior(N_GEN, seed=1)
    else:
        # L'AE n'a pas de sample_prior: son espace latent n'a aucune loi de reference.
        # On lui applique N(0,I) quand meme, pour montrer justement que rien ne l'y attache.
        rng = np.random.default_rng(1)
        codes_prior = Latent(array=rng.standard_normal((N_GEN, 2)).astype(np.float32),
                             nature="continuous")
    lignes.append(modele.decode(codes_prior))
    etiquettes.append(f"{nom}\nprior N(0,I)")

show_labeled_image_rows(
    lignes, MNIST_SHAPE, etiquettes, n=N_GEN,
    title="Generation: gaussienne ajustee vs prior N(0,I)",
    config=describe_vae(vae_model, n_train=len(X_tr), omit=("loss",)),
)

# Chiffre a l'appui: quelle part des tirages N(0,I) tombe dans la zone reellement occupee
# par les codes de chaque modele (boite englobante du nuage d'evaluation) ?
rng = np.random.default_rng(0)
tirages = rng.standard_normal((2000, 2)).astype(np.float32)
for nom, codes in [("AE", ae_latent.array), ("VAE", vae_stats["mu"])]:
    lo, hi = codes.min(axis=0), codes.max(axis=0)
    dedans = np.mean(np.all((tirages >= lo) & (tirages <= hi), axis=1))
    print(f"{nom:>4} | tirages N(0,I) tombant dans la zone occupee par les codes: {100 * dedans:5.1f}%")

In [ ]:
# Interpolation: relier deux images reelles par une droite dans le latent.
fig_lignes, fig_etiquettes = [], []
for nom, modele, latent in [("AE", ae_model, ae_latent), ("VAE", vae_model, vae_latent)]:
    chemin = interpolate_latent(latent.array[0], latent.array[1], steps=10)
    fig_lignes.append(modele.decode(Latent(array=chemin, nature="continuous")))
    fig_etiquettes.append(nom)

show_labeled_image_rows(
    fig_lignes, MNIST_SHAPE, fig_etiquettes, n=10,
    title="Interpolation entre deux images reelles (AE vs VAE)",
    config=describe_vae(vae_model, n_train=len(X_tr), omit=("loss",)),
)

In [ ]:
# Balayage de la variete 2D. La grille du VAE se lit sur les QUANTILES DU PRIOR (+/- 2.5
# ecarts-types couvrent ~99% de sa masse), pas sur le min/max des codes: c'est justement ce
# qu'apporte le VAE, on sait d'avance ou regarder. L'AE, lui, n'a que sa boite englobante.
GRILLE = 12

lo_ae, hi_ae = ae_latent.array.min(axis=0), ae_latent.array.max(axis=0)
grilles = {
    "AE (min/max des codes)": (np.linspace(lo_ae[0], hi_ae[0], GRILLE),
                               np.linspace(hi_ae[1], lo_ae[1], GRILLE), ae_model),
    "VAE (quantiles du prior)": (np.linspace(-2.5, 2.5, GRILLE),
                                 np.linspace(2.5, -2.5, GRILLE), vae_model),
}
for nom, (xs, ys, modele) in grilles.items():
    grille = np.array([[x, y] for y in ys for x in xs], dtype=np.float32)
    variete = modele.decode(Latent(array=grille, nature="continuous"))
    show_image_grid(variete, MNIST_SHAPE, nrow=GRILLE, ncol=GRILLE,
                    title=f"Variete du plan latent - {nom}",
                    config=describe_vae(vae_model, n_train=len(X_tr), omit=("loss",)))

## Le piege de la reduction est un bug beta=784

Le point le plus facile a rater de toute l'implementation. `nn.MSELoss()` et `nn.BCELoss()`
utilisent par defaut `reduction="mean"`, c'est-a-dire la moyenne sur le batch **et sur les 784
dimensions de l'image**. Or le KL, lui, **somme** sur les dimensions latentes. Les deux termes
ne sont alors plus homogenes: la reconstruction est 784 fois trop petite.

Et ce n'est pas un reglage approximatif, c'est une equivalence exacte. Adam est asymptotiquement
invariant d'echelle (`m / (sqrt(v) + eps)`), donc multiplier toute la perte par une constante ne
change rien: seul le **rapport** reconstruction / KL compte. Or

    recon_mean + 1 * KL   ==  (x784)   recon_sum + 784 * KL

**Utiliser `reduction="mean"` avec beta=1, c'est donc exactement entrainer avec beta=784.** Le KL
ecrase tout, l'encodeur renonce a encoder quoi que ce soit et rend le prior tel quel: `mu -> 0`,
`sigma -> 1`, `KL -> 0`. C'est le *posterior collapse*.

C'est pourquoi `VariationalAutoEncoder.fit` reecrit entierement `fit` au lieu d'appeler celui du
parent, et force `self.fonction_loss(reduction="sum")`. On reproduit ici le bug volontairement,
via beta=784 - ce qui prouve au passage l'equivalence.

In [ ]:
BETA_COLLAPSE = 784.0   # == reduction="mean" avec beta=1

collapse_run = run_vae_experiment(X_tr, X_mnist_eval, 784, 2, nn.ReLU, EPOCHS_SWEEP,
                                  beta=BETA_COLLAPSE, seed=GRAINE)
collapse_stats = posterior_stats(collapse_run["model"], X_mnist_eval)

print(f"{'':>10} | {'KL total':>9} | {'mu.std par dim':>22} | {'sigma par dim':>20} | {'dims actives':>12}")
for nom, stats in [("beta=1", vae_stats), (f"beta={BETA_COLLAPSE:.0f}", collapse_stats)]:
    print(f"{nom:>10} | {stats['kl_total']:>9.4f} | {str(np.round(stats['mu'].std(axis=0), 4)):>22} "
          f"| {str(np.round(stats['sigma'].mean(axis=0), 3)):>20} | {stats['dims_actives']:>12}")
print("\nAu collapse: sigma vaut exactement 1 (le prior), mu ne bouge plus, KL = 0.")
print("L'encodeur a cesse d'encoder: le decodeur recoit du bruit pur et rend l'image moyenne.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
for ax, (nom, stats) in zip(axes, [("beta=1 (sain)", vae_stats),
                                   (f"beta={BETA_COLLAPSE:.0f} (collapse)", collapse_stats)]):
    codes = stats["mu"]
    scatter = ax.scatter(codes[:, 0], codes[:, 1], c=y_mnist_eval, cmap="tab10", s=4, alpha=0.6)
    ax.set_title(f"{nom} - KL={stats['kl_total']:.3f}", fontsize=10)
    ax.set_xlabel("z1"); ax.set_ylabel("z2")
fig.colorbar(scatter, ax=list(axes), label="chiffre")
finish_figure(fig, "Posterior collapse: l'espace latent se reduit a un point",
              describe_vae(vae_model, n_train=len(X_tr), omit=("loss",)), layout=False)

# Toutes les images generees deviennent identiques: c'est la signature visuelle du collapse.
show_labeled_image_rows(
    [vae_model.decode(vae_model.sample_prior(8, seed=1)),
     collapse_run["model"].decode(collapse_run["model"].sample_prior(8, seed=1))],
    MNIST_SHAPE, ["beta=1 (sain)", f"beta={BETA_COLLAPSE:.0f} (collapse)"], n=8,
    title="Generation au collapse: le decodeur rend la meme image quel que soit z",
    config=describe_vae(collapse_run["model"], n_train=len(X_tr), omit=("loss",)),
)

## Experimentation - beta

`beta` regle l'arbitrage entre reconstruction et KL, donc entre **fidelite** et **conformite au
prior**. Le balayage relie les deux bouts du notebook:

- **beta=0**: plus aucune pression vers le prior. Le latent redevient un nuage libre, comme
  celui d'un AutoEncoder.
- **beta=1**: l'ELBO standard.
- **beta > 1** (beta-VAE): on paie de la fidelite pour un latent plus propre et plus demele; des
  dimensions commencent a mourir.
- **beta=784**: le bug de la reduction, reinjecte ici comme dernier point du balayage.

**beta=0 n'est pas exactement un AutoEncoder**, et il faut le dire: la reparametrisation injecte
toujours du bruit (le modele apprend a l'eteindre, `sigma -> ~0.01`, mais c'est un AE *bruite
converge*, pas un AE deterministe); l'encodeur porte une tete `logvar` en plus; et la
reconstruction est x784 par rapport a la moyenne de l'AE (quasi neutre sous Adam). beta=0 retrouve
le **comportement** de l'AE, pas l'AE.

**Un piege de lecture sur `||Cov(mu) - I||_F`.** Cette mesure porte sur le nuage des `mu`, pas
sur les `z` reellement tires. Au collapse, `mu -> 0` donc `Cov(mu) -> 0`, et l'ecart tend vers
`||0 - I||_F = sqrt(latent_dim)`, soit **exactement 1.414 a latent 2**. Ce n'est donc pas un
mauvais accord au prior: c'est le **plancher de la metrique quand mu ne transporte plus aucune
information**. Les `z = mu + sigma*eps` d'un modele collapse suivent au contraire `N(0,I)` a la
perfection - ils *sont* le prior. La courbe n'est pas monotone en beta pour cette raison, et
c'est le nombre de **dimensions actives** qui tranche: 2 a beta=1, 0 au collapse.

In [ ]:
BETAS = [0.0, 0.1, 0.5, 1.0, 4.0, 10.0, 784.0]

beta_runs = {}
print(f"{'beta':>6} | {'MSE eval':>9} | {'KL total':>9} | {'||Cov-I||_F':>12} | {'dims actives':>12}")
for beta in BETAS:
    run = run_vae_experiment(X_tr, X_mnist_eval, 784, 2, nn.ReLU, EPOCHS_SWEEP,
                             beta=beta, seed=GRAINE)
    stats = posterior_stats(run["model"], X_mnist_eval)
    _, ecart = prior_gap(stats["mu"])
    beta_runs[beta] = {"run": run, "stats": stats, "gap": ecart}
    print(f"{beta:>6.1f} | {run['report']['reconstruction_mse']:>9.4f} | {stats['kl_total']:>9.4f} "
          f"| {ecart:>12.3f} | {stats['dims_actives']:>12}")

# Explore ici: beta (axe x) -> retire de la note, qui porterait sinon la valeur d'un seul run.
beta_config = describe_vae(vae_model, n_train=len(X_tr), omit=("loss",),
                           extra=f"loss={vae_model.fonction_loss.__name__}(sum)+beta*KL")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
mses = [beta_runs[b]["run"]["report"]["reconstruction_mse"] for b in BETAS]
kls = [beta_runs[b]["stats"]["kl_total"] for b in BETAS]
gaps = [beta_runs[b]["gap"] for b in BETAS]
x = np.arange(len(BETAS))

axes[0].plot(x, mses, marker="o", color="tab:blue", label="MSE d'evaluation")
axes[0].set_ylabel("MSE d'evaluation", color="tab:blue")
axes[0].tick_params(axis="y", labelcolor="tab:blue")
jumeau = axes[0].twinx()
jumeau.plot(x, kls, marker="s", color="tab:orange", label="KL total")
jumeau.set_ylabel("KL total (nats)", color="tab:orange")
jumeau.tick_params(axis="y", labelcolor="tab:orange")
axes[0].set_xticks(x, [f"{b:g}" for b in BETAS])
axes[0].set_xlabel("beta"); axes[0].set_title("Fidelite vs conformite au prior")

axes[1].plot(x, gaps, marker="o", color="tab:green")
axes[1].axhline(0, color="0.6", linestyle="--", linewidth=1)
axes[1].set_xticks(x, [f"{b:g}" for b in BETAS])
axes[1].set_xlabel("beta"); axes[1].set_ylabel("||Cov(mu) - I||_F")
axes[1].set_title("Distance du latent au prior (0 = parfait)")
finish_figure(fig, "MNIST: effet de beta", beta_config)

show_labeled_image_rows(
    [beta_runs[b]["run"]["model"].decode(beta_runs[b]["run"]["model"].sample_prior(8, seed=1))
     for b in BETAS],
    MNIST_SHAPE, [f"beta={b:g}" for b in BETAS], n=8,
    title="Generation depuis le prior N(0,I) selon beta",
    config=beta_config,
)

## Experimentation - dimension latente et collapse

Le balayage de `03` reprend a l'identique (memes dimensions, meme debit par image), avec deux
axes en plus que seul un VAE possede: le **KL** et le nombre de **dimensions actives**.

Une dimension est dite active si son KL depasse un seuil (~0.01 nat). En dessous, l'encodeur y
rend exactement le prior: elle ne transporte rien et le decodeur l'ignore. C'est le *posterior
collapse*, et c'est ce qui explique pourquoi la MSE d'un VAE **sature** quand on agrandit le
latent: les dimensions supplementaires ne servent tout simplement pas.

In [ ]:
latent_dims = [2, 8, 16, 32, 64]

vae_latent_runs = {}
print(f"{'latent':>6} | {'MSE eval':>9} | {'KL total':>9} | {'dims actives':>12} | {'ratio':>7}")
for latent_dim in latent_dims:
    run = run_vae_experiment(X_tr, X_mnist_eval, 784, latent_dim, nn.ReLU, EPOCHS_SWEEP,
                             beta=1.0, seed=GRAINE)
    stats = posterior_stats(run["model"], X_mnist_eval)
    vae_latent_runs[latent_dim] = {"run": run, "stats": stats}
    print(f"{latent_dim:>6} | {run['report']['reconstruction_mse']:>9.4f} | {stats['kl_total']:>9.3f} "
          f"| {stats['dims_actives']:>4} / {latent_dim:<5} | {run['report']['compression_ratio']:>7.2f}")

latent_config = describe_vae(vae_latent_runs[2]["run"]["model"], n_train=len(X_tr),
                             omit=("couches",), extra="couches=geomspace(784->latent, 3+3)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].plot(latent_dims, [vae_latent_runs[d]["run"]["report"]["reconstruction_mse"] for d in latent_dims],
             marker="o", label="VAE")
axes[0].set_xlabel("dimension latente"); axes[0].set_ylabel("MSE d'evaluation")
axes[0].set_title("Qualite vs dimension latente"); axes[0].legend()

axes[1].plot(latent_dims, [vae_latent_runs[d]["stats"]["dims_actives"] for d in latent_dims],
             marker="o", label="dimensions actives (KL > 0.01)")
axes[1].plot(latent_dims, latent_dims, linestyle="--", color="0.6", label="si toutes servaient")
axes[1].set_xlabel("dimension latente"); axes[1].set_ylabel("nb de dimensions")
axes[1].set_title("Combien de dimensions servent vraiment ?"); axes[1].legend()
finish_figure(fig, "MNIST: dimension latente et collapse", latent_config)

show_labeled_image_rows(
    [X_mnist_eval] + [vae_latent_runs[d]["run"]["reconstruction"] for d in latent_dims],
    MNIST_SHAPE,
    ["original"] + [f"latent={d}\n{vae_latent_runs[d]['run']['report']['reconstruction_mse']:.4f}"
                    for d in latent_dims],
    n=8, title="VAE: decompression selon la dimension latente (etiquette = MSE)",
    config=latent_config,
)

In [ ]:
# Le collapse rendu visible: KL dimension par dimension, a latent 16.
DIM_COLLAPSE = 16
stats16 = vae_latent_runs[DIM_COLLAPSE]["stats"]
kl_tri = np.sort(stats16["kl_per_dim"])[::-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
couleurs = ["tab:blue" if k > 0.01 else "0.7" for k in kl_tri]
axes[0].bar(range(DIM_COLLAPSE), kl_tri, color=couleurs)
axes[0].axhline(0.01, color="tab:red", linestyle="--", linewidth=1, label="seuil actif (0.01 nat)")
axes[0].set_xlabel("dimension latente (triee par KL)"); axes[0].set_ylabel("KL (nats)")
axes[0].set_title(f"{stats16['dims_actives']}/{DIM_COLLAPSE} dimensions actives"); axes[0].legend()

axes[1].bar(range(DIM_COLLAPSE), np.sort(stats16["sigma"].mean(axis=0))[::-1], color="tab:orange")
axes[1].axhline(1.0, color="tab:red", linestyle="--", linewidth=1, label="sigma=1 (prior: dim morte)")
axes[1].set_xlabel("dimension latente (triee)"); axes[1].set_ylabel("sigma moyen")
axes[1].set_title("Une dimension morte rend exactement le prior"); axes[1].legend()
finish_figure(fig, f"VAE: posterior collapse a latent={DIM_COLLAPSE}",
              describe_vae(vae_latent_runs[DIM_COLLAPSE]["run"]["model"], n_train=len(X_tr)))

## Experimentation - perte x activation de sortie

Meme matrice que dans `03`, et **memes deux cases impossibles**: le garde-fou BCE est herite
d'`AutoEncoder`, puisque le decodeur est le meme. `BCELoss` tolere exactement 0 et 1 mais rejette
toute valeur superieure a 1; une sortie non bornee (`ReLU`, ou aucune activation) declenche donc
une assertion en cours d'entrainement - cote device sur GPU, ce qui corrompt le contexte CUDA
pour toutes les cellules suivantes. On saute ces couples explicitement.

Rappel VAE: la reconstruction est ici sommee sur les dimensions (`reduction="sum"`), sinon on
retomberait sur le collapse. Et pour des pixels dans [0,1] derriere une Sigmoid, `BCE(sum)` est
la log-vraisemblance de Bernoulli, donc le terme de reconstruction naturel de l'ELBO.

In [ ]:
loss_functions = {"MSE": nn.MSELoss, "L1": nn.L1Loss, "BCE": nn.BCELoss}
output_activations = {"Sigmoid": nn.Sigmoid, "ReLU": nn.ReLU, "aucune": None}
COUPLES_INVALIDES = {("ReLU", "BCE"), ("aucune", "BCE")}
OUT_LOSS_LATENT = 32

vae_out_loss = {}
for out_name, out_act in output_activations.items():
    for loss_name, loss_cls in loss_functions.items():
        if (out_name, loss_name) in COUPLES_INVALIDES:
            print(f"out_act={out_name:>8} | perte={loss_name:<3} | INVALIDE (sortie non bornee + BCE)")
            continue
        run = run_vae_experiment(X_tr, X_mnist_eval, 784, OUT_LOSS_LATENT, nn.ReLU, EPOCHS_SWEEP,
                                 loss_function=loss_cls, output_activation_function=out_act,
                                 beta=1.0, seed=GRAINE)
        stats = posterior_stats(run["model"], X_mnist_eval)
        vae_out_loss[(out_name, loss_name)] = {"run": run, "stats": stats}
        print(f"out_act={out_name:>8} | perte={loss_name:<3} | MSE={run['report']['reconstruction_mse']:.4f} "
              f"| KL={stats['kl_total']:>7.3f} | dims actives={stats['dims_actives']:>2}/{OUT_LOSS_LATENT}")

out_loss_config = describe_vae(vae_out_loss[("Sigmoid", "MSE")]["run"]["model"],
                               n_train=len(X_tr), omit=("loss", "out_act"))

matrice = np.full((len(output_activations), len(loss_functions)), np.nan)
for i, out_name in enumerate(output_activations):
    for j, loss_name in enumerate(loss_functions):
        if (out_name, loss_name) in vae_out_loss:
            matrice[i, j] = vae_out_loss[(out_name, loss_name)]["run"]["report"]["reconstruction_mse"]

fig, ax = plt.subplots(figsize=(5.5, 3.6))
image = ax.imshow(np.ma.masked_invalid(matrice), cmap="viridis_r")
image.cmap.set_bad("0.85")
ax.set_xticks(range(len(loss_functions)), list(loss_functions))
ax.set_yticks(range(len(output_activations)), list(output_activations))
ax.set_xlabel("terme de reconstruction"); ax.set_ylabel("activation de sortie")
for i, out_name in enumerate(output_activations):
    for j, loss_name in enumerate(loss_functions):
        valeur = matrice[i, j]
        ax.text(j, i, "invalide" if np.isnan(valeur) else f"{valeur:.4f}",
                ha="center", va="center", fontsize=9,
                color="0.35" if np.isnan(valeur) else "white")
fig.colorbar(image, ax=ax, label="MSE d'evaluation")
finish_figure(fig, "VAE: MSE d'evaluation selon sortie x reconstruction", out_loss_config)

## Experimentation - nombre d'epochs

Comme dans `03`, chaque budget entraine un modele independant (`fit` reinstancie son optimiseur
Adam a chaque appel, donc 4 x 20 epochs ne valent pas 80).

En plus de l'AE: on suit **separement** reconstruction et KL. C'est plus informatif qu'une perte
totale, parce que les deux termes ne progressent pas ensemble - le KL monte souvent d'abord (le
modele apprend a se servir du latent), puis se stabilise pendant que la reconstruction continue
de descendre.

In [ ]:
EPOCH_BUDGETS = [5, 10, 20, 40, 80]

vae_epoch_runs = {}
print(f"{'epochs':>6} | {'MSE train':>9} | {'MSE eval':>9} | {'ecart':>7} | {'KL total':>9}")
for budget in EPOCH_BUDGETS:
    run = run_vae_experiment(X_tr, X_mnist_eval, 784, 2, nn.ReLU, budget, beta=1.0, seed=GRAINE)
    # MSE d'entrainement, comparable a la MSE d'evaluation (la perte BCE+KL ne l'est pas).
    train_rec = run["model"].decode(run["model"].encode(X_tr))
    run["train_mse"] = float(np.mean((X_tr - train_rec) ** 2))
    stats = posterior_stats(run["model"], X_mnist_eval)
    vae_epoch_runs[budget] = {"run": run, "stats": stats}
    print(f"{budget:>6} | {run['train_mse']:>9.4f} | {run['report']['reconstruction_mse']:>9.4f} "
          f"| {run['report']['reconstruction_mse'] - run['train_mse']:>+7.4f} | {stats['kl_total']:>9.3f}")

epochs_config = describe_vae(vae_epoch_runs[EPOCH_BUDGETS[-1]]["run"]["model"],
                             n_train=len(X_tr), omit=("epochs",))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].plot(EPOCH_BUDGETS, [vae_epoch_runs[b]["run"]["train_mse"] for b in EPOCH_BUDGETS],
             marker="o", label="MSE d'entrainement")
axes[0].plot(EPOCH_BUDGETS, [vae_epoch_runs[b]["run"]["report"]["reconstruction_mse"] for b in EPOCH_BUDGETS],
             marker="o", label="MSE d'evaluation")
axes[0].set_xscale("log"); axes[0].set_xticks(EPOCH_BUDGETS, [str(b) for b in EPOCH_BUDGETS])
axes[0].set_xlabel("nombre d'epochs"); axes[0].set_ylabel("MSE"); axes[0].legend()
axes[0].set_title("Sous- ou sur-apprentissage")

plus_long = vae_epoch_runs[EPOCH_BUDGETS[-1]]["run"]["model"]
axes[1].plot(plus_long.recon_history, label="reconstruction (BCE somme)", color="tab:blue")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("BCE par image", color="tab:blue")
axes[1].tick_params(axis="y", labelcolor="tab:blue")
jumeau = axes[1].twinx()
jumeau.plot(plus_long.kl_history, label="KL", color="tab:orange")
jumeau.set_ylabel("KL (nats)", color="tab:orange")
jumeau.tick_params(axis="y", labelcolor="tab:orange")
axes[1].set_title(f"Les deux termes ({EPOCH_BUDGETS[-1]} epochs)")
finish_figure(fig, "VAE: qualite vs budget d'entrainement", epochs_config)

show_labeled_image_rows(
    [vae_epoch_runs[b]["run"]["model"].decode(vae_epoch_runs[b]["run"]["model"].sample_prior(8, seed=1))
     for b in EPOCH_BUDGETS],
    MNIST_SHAPE, [f"epochs={b}" for b in EPOCH_BUDGETS], n=8,
    title="VAE: generation depuis le prior selon le budget d'entrainement",
    config=epochs_config,
)

## Experimentation - profondeur du reseau a latent fixe

Meme protocole que dans `03`: latent fixe a 2, profondeur de 1 a 6, deux modes (symetrique et
encodeur seul), 3 graines et barres d'erreur - une seule graine invente des accidents.

Rappel du point cle: `get_codebook()` ne renvoie que les poids du **decodeur**. Approfondir
l'encodeur ne coute donc rien en transmission (il n'est jamais envoye), alors qu'approfondir le
decodeur fait exploser le codebook.

La profondeur 1 est le premier vrai consommateur du garde-fou de `VariationalEncoder`: le tronc
y est **vide** et les deux tetes lisent directement les 784 pixels.

In [ ]:
DEPTHS = [1, 2, 3, 4, 5, 6]
DEPTH_SEEDS = [0, 1, 2]
DEPTH_MODES = {"symetrique": None, "encodeur seul": 3}

vae_depth = {}
for mode_name, decodeur_fixe in DEPTH_MODES.items():
    for depth in DEPTHS:
        mses, kls, report = [], [], None
        for seed in DEPTH_SEEDS:
            run = run_vae_experiment(
                X_tr, X_mnist_eval, 784, 2, nn.ReLU, EPOCHS_SWEEP, beta=1.0, seed=seed,
                encoder_layer_num=depth,
                decoder_layer_num=depth if decodeur_fixe is None else decodeur_fixe,
            )
            mses.append(run["report"]["reconstruction_mse"])
            kls.append(posterior_stats(run["model"], X_mnist_eval)["kl_total"])
            report = run["report"]
        vae_depth[(mode_name, depth)] = {
            "mse_moyenne": float(np.mean(mses)), "mse_ecart": float(np.std(mses)),
            "kl_moyen": float(np.mean(kls)),
            "codebook": report["codebook_bytes"], "total": report["total_compressed_bytes"],
            "ratio": report["compression_ratio"],
        }
        r = vae_depth[(mode_name, depth)]
        print(f"{mode_name:>13} | d={depth} | MSE={r['mse_moyenne']:.4f} +/- {r['mse_ecart']:.4f} "
              f"| KL={r['kl_moyen']:>6.3f} | codebook={r['codebook']:>9,} o | ratio={r['ratio']:6.2f}")
    print()

meilleure = min(DEPTHS, key=lambda d: vae_depth[("symetrique", d)]["mse_moyenne"])
print(f"Meilleure profondeur (symetrique, MSE moyenne): d={meilleure}")

# `couches` decrit l'encodeur seul: il induirait en erreur en mode "encodeur seul".
depth_config = describe_vae(
    vae_model, n_train=len(X_tr), omit=("couches", "epochs"),
    extra=f"encodeur=geomspace(784->2, d) | latent=2 | epochs={EPOCHS_SWEEP} | {len(DEPTH_SEEDS)} graines",
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for mode_name in DEPTH_MODES:
    moyennes = [vae_depth[(mode_name, d)]["mse_moyenne"] for d in DEPTHS]
    ecarts = [vae_depth[(mode_name, d)]["mse_ecart"] for d in DEPTHS]
    totaux = [vae_depth[(mode_name, d)]["total"] for d in DEPTHS]
    axes[0].errorbar(DEPTHS, moyennes, yerr=ecarts, marker="o", capsize=3, label=mode_name)
    axes[1].errorbar(totaux, moyennes, yerr=ecarts, marker="o", capsize=3, label=mode_name)
axes[0].axvline(meilleure, color="0.6", linestyle="--", linewidth=1)
axes[0].set_xlabel("profondeur (nb de couches)"); axes[0].set_ylabel("MSE d'evaluation")
axes[0].set_title("Qualite vs profondeur (moyenne +/- ecart-type)"); axes[0].legend()
axes[1].set_xscale("log")
axes[1].set_xlabel("taille compressee totale (octets, echelle log)"); axes[1].set_ylabel("MSE d'evaluation")
axes[1].set_title("Compromis taille / qualite"); axes[1].legend()
finish_figure(fig, "VAE: profondeur a dimension latente fixee (2)", depth_config)

## Pourquoi il n'y a pas d'experience sur l'activation latente

Le notebook `03` consacre une section entiere a `latent_activation` (latent lineaire / ReLU /
Sigmoid). Ce parametre n'est **volontairement pas expose** par `VariationalAutoEncoder`, parce
qu'il n'a pas de sens ici - et c'est instructif de comprendre pourquoi.

Dans un AutoEncoder, le latent **est** le code: le borner est un choix de representation
discutable mais coherent. Dans un VAE, `mu` n'est pas un code, c'est un **parametre de
distribution**, et le KL le compare a `N(0, I)`. Le borner revient a se battre contre le KL:

- `Sigmoid(mu)` vit dans `(0,1)`. Une gaussienne de moyenne confinee dans `(0,1)` ne peut jamais
  coincider avec `N(0,1)`: le KL a un **plancher structurel non nul**, le modele optimise une
  borne qu'il ne peut pas atteindre.
- `ReLU(mu) >= 0` rend **la moitie de la masse du prior inatteignable** par l'encodeur - or
  `sample_prior` tire justement des z negatifs. La generation est cassee par construction:
  le decodeur recevrait des codes que l'encodeur ne produit jamais. C'est exactement le defaut
  de l'AutoEncoder que le VAE est cense supprimer.
- Sur `logvar`, ce serait pire encore: `ReLU(logvar) >= 0` impose `sigma >= 1` partout, soit un
  plancher de bruit qui interdit a toute dimension de transporter de l'information precise.

De meme, le **masque de neurones morts** de `03` (sortie ReLU) n'est pas repris: il porte sur le
**decodeur**, qui est rigoureusement le meme objet ici. Le resultat serait identique, se reporter
a `03_autoencoder`.

## Partie B - shapes

In [ ]:
X_shapes_train, y_shapes_train, shape_names = load_shapes_npz(split="train", max_samples=12000)
X_shapes_eval, y_shapes_eval, _ = load_shapes_npz(split="validation", max_samples=3000)
X_shapes_train = X_shapes_train.reshape(len(X_shapes_train), -1)
X_shapes_eval = X_shapes_eval.reshape(len(X_shapes_eval), -1)

SHAPES_SHAPE = (3, 32, 32)
SHAPES_DIM = 3 * 32 * 32
X_shapes_tr, _ = subsample_dataset(X_shapes_train, y_shapes_train, 6000, seed=1)
print("train:", X_shapes_train.shape, "| eval:", X_shapes_eval.shape, "| classes:", shape_names)

shapes_vae_run = run_vae_experiment(X_shapes_tr, X_shapes_eval, SHAPES_DIM, 2, nn.ReLU, EPOCHS,
                                    beta=1.0, seed=GRAINE)
shapes_vae = shapes_vae_run["model"]
shapes_stats = posterior_stats(shapes_vae, X_shapes_eval)
shapes_centre, shapes_gap = prior_gap(shapes_stats["mu"])

print_compression_report(shapes_vae_run["report"])
print(f"\n||E[mu]||={shapes_centre:.3f} | ||Cov-I||_F={shapes_gap:.3f} | KL={shapes_stats['kl_total']:.3f}")

SHAPES_CONFIG = describe_vae(shapes_vae, n_train=len(X_shapes_tr))

show_original_vs_reconstruction_grid(X_shapes_eval, shapes_vae_run["reconstruction"], SHAPES_SHAPE,
                                     n=8, title="Shapes: original (haut) vs reconstruit (bas) - VAE",
                                     config=SHAPES_CONFIG)

plot_latent_scatter(shapes_stats["mu"], y_shapes_eval, class_names=shape_names,
                    title=f"Shapes: espace latent 2D du VAE (||Cov-I||={shapes_gap:.2f})",
                    config=SHAPES_CONFIG)

show_labeled_image_rows(
    [generate_from_latent_using_gaussian(shapes_vae, shapes_vae_run["latent"], 8, seed=1),
     shapes_vae.decode(shapes_vae.sample_prior(8, seed=1))],
    SHAPES_SHAPE, ["gaussienne ajustee", "prior N(0,I)"], n=8,
    title="Shapes: generer avec ou sans ajustement (VAE)",
    config=SHAPES_CONFIG,
)

## Reponses aux questions du projet

**1. Nature de l'espace latent.**

Continu, comme pour l'AutoEncoder (`encode` renvoie un `Latent` de nature `"continuous"`), mais
avec une difference decisive: sa **loi est connue d'avance**. L'encodeur ne rend pas un point
mais une distribution `q(z|x) = N(mu, diag(exp(logvar)))`, et le terme KL la contraint a
ressembler au prior `N(0, I)`. La ou l'AutoEncoder produisait un nuage de forme et d'echelle
arbitraires - qu'il fallait mesurer apres coup pour esperer y echantillonner - le VAE garantit
la forme par construction.

**2. Codebook.**

Identique a l'AutoEncoder: `get_codebook()` renvoie les poids du **decodeur**, herites sans
modification. La tete `logvar` de l'encodeur n'en fait pas partie et n'est jamais transmise: le
recepteur n'a besoin que de savoir transformer un code en image. Le VAE ne change donc rien au
cout du dictionnaire partage.

**3. Qualite de reconstruction.**

A dimension latente egale, elle est **moins bonne** que celle de l'AutoEncoder, et c'est normal:
une partie de la capacite paie le KL, et le bruit de la reparametrisation force le decodeur a
rester coherent dans un voisinage de chaque code plutot que sur un point isole. C'est un
**echange**, pas une regression: on perd de la fidelite, on gagne un espace ou l'on peut
echantillonner et interpoler sans tomber dans le vide. `beta` est le curseur exact de cet
echange, et `compression_report` n'en voit qu'un cote.

**4. Code byte (taille du message).**

Strictement identique a l'AutoEncoder: `dimension_latente x 4` octets par image (on transmet
`mu`, en float32), plus le codebook une seule fois. Les deux modeles sont donc comparables a
**debit egal** - toute difference observee vient de ce qu'ils font de ce debit, pas de sa taille.
Le balayage de la dimension latente montre d'ailleurs que le VAE **n'utilise pas tout** le
budget qu'on lui donne: passe un certain point, les dimensions supplementaires collapsent et le
debit reel stagne, quand bien meme on continue de payer les octets.